In [0]:
# ============================================================
# NOTEBOOK 02 — TRUSTED: FEATURE ENGINEERING
# Vermont School - Early Warning System V2
#
# INPUT:  bronze/prepared/24_25/ and bronze/prepared/25_26/
# OUTPUT: trusted/train_dataset/   ← 24-25 (T1+T2 features, risk_level target)
#         trusted/predict_dataset/ ← 25-26 (T1+T2 features only)
# ============================================================

BRONZE  = "/Volumes/workspace/vermont/bronze"
TRUSTED = "/Volumes/workspace/vermont/trusted"
SILVER  = "/Volumes/workspace/vermont/silver"

PREP_24 = f"{BRONZE}/prepared/24_25"
PREP_25 = f"{BRONZE}/prepared/25_26"

TRAIN_PATH   = f"{TRUSTED}/train_dataset"
PREDICT_PATH = f"{TRUSTED}/predict_dataset"

for path in [TRAIN_PATH, PREDICT_PATH]:
    dbutils.fs.mkdirs(path)

# Subject groups
GROUPS = [
    'Science', 'I_and_S', 'Mathematics', 'English',
    'Lengua_Castellana', 'Mandarin', 'Financial_Maths',
    'ICT_STEM', 'Physical_Education', 'Research_Methodology'
]

# Features used for training — only T1 and T2
# (what we know MID-YEAR, before final grades)
FEATURES_T1_T2 = (
    [f'{g}_T1' for g in GROUPS] +
    [f'{g}_T2' for g in GROUPS] +
    ['total_absences', 'absence_class', 'absence_school',
     'late', 'early_leave', 'n_f1', 'n_f2', 'n_fault_types']
)

TARGET = 'risk_level'

print("✓ Configuration loaded")
print(f"  Features (T1+T2 only): {len(FEATURES_T1_T2)}")
print(f"  Target: {TARGET}")
print(f"\n  Key design decision:")
print(f"  → Model trained with T1+T2 only (mid-year snapshot)")
print(f"  → Target is FINAL risk level (end of year)")
print(f"  → This enables genuine early warning predictions")

In [0]:
# CELDA 2 — Build training and prediction datasets

import pandas as pd
import numpy as np

# Load both years
df_24 = spark.read.parquet(PREP_24).toPandas()
df_25 = spark.read.parquet(PREP_25).toPandas()

print(f"Loaded 24-25: {len(df_24)} students")
print(f"Loaded 25-26: {len(df_25)} students")

# ── TRAINING DATASET (24-25) ──
# Features: T1 + T2 grades + attendance + disciplinary
# Target: final risk level (calculated from cumulative grades)

# Check available features
missing_train = [f for f in FEATURES_T1_T2 if f not in df_24.columns]
if missing_train:
    print(f"\n  Warning — missing features in 24-25: {missing_train}")

available_features = [f for f in FEATURES_T1_T2 if f in df_24.columns]

# Build training set
df_train = df_24[['student_id', 'grade', 'section_anon', 'year'] + 
                  available_features + [TARGET]].copy()

# Fill NaN with 0 for disciplinary, keep NaN for grades
disc_cols = ['total_absences','absence_class','absence_school',
             'late','early_leave','n_f1','n_f2','n_fault_types']
df_train[disc_cols] = df_train[disc_cols].fillna(0)

print(f"\n── TRAINING DATASET (24-25) ──")
print(f"  Students: {len(df_train)}")
print(f"  Features: {len(available_features)}")
print(f"  Target distribution:")
print(df_train[TARGET].value_counts().to_string())

# ── PREDICTION DATASET (25-26) ──
# Same features, T1 + T2 only
# Also include T3 partial for alert confirmation later

t3_cols = [f'{g}_T3' for g in GROUPS if f'{g}_T3' in df_25.columns]

df_predict = df_25[['student_id', 'grade', 'section_anon', 'year'] +
                    available_features + t3_cols +
                    ['risk_level', 'avg_cumulative', 
                     'min_cumulative', 'n_subjects_below_4']].copy()

df_predict[disc_cols] = df_predict[disc_cols].fillna(0)

print(f"\n── PREDICTION DATASET (25-26) ──")
print(f"  Students: {len(df_predict)}")
print(f"  Features: {len(available_features)}")
print(f"  T3 partial columns included: {len(t3_cols)}")
print(f"  (T3 used only for alert confirmation, NOT for prediction)")

# ── SAVE TO TRUSTED ──
spark.createDataFrame(df_train).write.mode("overwrite").parquet(TRAIN_PATH)
spark.createDataFrame(df_predict).write.mode("overwrite").parquet(PREDICT_PATH)

print(f"\n✓ Saved to Trusted:")
print(f"  {TRAIN_PATH}")
print(f"  {PREDICT_PATH}")

In [0]:
# CELDA 3 — Validation and summary

df_train_check   = spark.read.parquet(TRAIN_PATH).toPandas()
df_predict_check = spark.read.parquet(PREDICT_PATH).toPandas()

print("=" * 55)
print("TRUSTED DATASETS — VALIDATION SUMMARY")
print("=" * 55)

# Check nulls in key features
print("\n── Training dataset nulls (key features) ──")
key_features = [f'{g}_T1' for g in GROUPS[:5]] + \
               [f'{g}_T2' for g in GROUPS[:5]] + \
               ['total_absences', 'n_f1', 'n_f2']
nulls = df_train_check[key_features].isnull().sum()
print(nulls[nulls > 0].to_string() if nulls[nulls > 0].any() 
      else "  No nulls in key features ✓")

print("\n── Prediction dataset nulls (key features) ──")
nulls2 = df_predict_check[key_features].isnull().sum()
print(nulls2[nulls2 > 0].to_string() if nulls2[nulls2 > 0].any() 
      else "  No nulls in key features ✓")

# Grade distribution
print("\n── Students per grade ──")
print("  Training (24-25):")
print(df_train_check['grade'].value_counts().sort_index().to_string())
print("  Prediction (25-26):")
print(df_predict_check['grade'].value_counts().sort_index().to_string())

# Feature preview
print(f"\n── Feature ranges (training) ──")
cols_check = ['Science_T1','Science_T2','Mathematics_T1',
              'Mathematics_T2','total_absences','n_f1']
print(df_train_check[cols_check].describe().round(2).to_string())

print(f"\n✓ Notebook 02 complete — Trusted datasets ready")
print(f"  Training:   {TRAIN_PATH}")
print(f"  Prediction: {PREDICT_PATH}")